# 03 · A máquina que entende **postura** — pose, braços e contagem de movimento

A demo mais forte do bloco, porque a plateia participa: você pede para a sala
levantar o braço e o número muda na tela.

O que ela mostra:
1. o **esqueleto** de cada pessoa (17 pontos), pintado **por parte do corpo** —
   cabeça, braços, tronco e pernas, cada uma com a sua cor
2. **quantas pessoas** estão na cena
3. **quantas com o braço para cima** e quantas para baixo
4. cada ciclo levantar→baixar contado como **uma movimentação**
5. pelos pontos da cabeça, quem está **de frente, de perfil ou de costas**

> Fala de palco: *"ninguém programou 'braço'. O modelo devolve pontos; a regra
> de negócio — o que conta como levantado — sou eu que escrevo. É essa a divisão
> de trabalho entre a IA e você."*

In [ ]:
# ── 1. instala a biblioteca e monta o Google Drive ──
%pip install -q ultralytics
from google.colab import drive
drive.mount('/content/drive')

from ultralytics import YOLO
import ultralytics, torch, os, glob
ultralytics.checks()
print("GPU disponivel:", torch.cuda.is_available())

# ── 2. a raiz de tudo, e a conferência de que ela é REAL ──────────────
#
# ARMADILHA que já custou uma sessão: a linha acima cria a variável
# `drive` (minúscula), que é o MÓDULO do Colab. Se algum caminho for
# escrito com `drive` em vez de `DRIVE`, o Python aceita numa boa e
# monta um caminho como
#     <module 'google.colab.drive' from '/usr/local/...'>/04-garrafas
# O código roda, cria pastas, exporta arquivos — tudo no disco
# temporário do Colab, que evapora quando a sessão encerra. Nada disso
# chega ao seu Drive, e não há erro nenhum na tela.
#
# A conferência abaixo transforma esse silêncio num aviso imediato.

DRIVE = "/content/drive/MyDrive/PALESTRA-IA"

if not DRIVE.startswith("/content/drive/"):
    raise SystemExit(
        "DRIVE aponta para fora do Google Drive: " + repr(DRIVE) + "\n"
        "Provavelmente algum caminho usou `drive` (o módulo) em vez de `DRIVE`.")
if not os.path.isdir("/content/drive/MyDrive"):
    raise SystemExit("O Drive não montou. Rode esta célula de novo e autorize o acesso.")

os.makedirs(DRIVE, exist_ok=True)
print("raiz no Drive:", DRIVE)
print("existe de verdade:", os.path.isdir(DRIVE))

In [ ]:
# ── ajuste de PALCO: tudo grande, porque a sala enxerga de 6 a 10 m ──
import matplotlib
matplotlib.rcParams.update({
    "figure.figsize": (16, 9),
    "figure.dpi": 110,
    "font.size": 22,
    "axes.titlesize": 30,
    "axes.labelsize": 24,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 22,
    "axes.grid": True,
    "grid.alpha": .25,
    "axes.facecolor": "#0d1117",
    "figure.facecolor": "#0d1117",
    "text.color": "#e6edf3",
    "axes.labelcolor": "#e6edf3",
    "xtick.color": "#e6edf3",
    "ytick.color": "#e6edf3",
    "axes.edgecolor": "#30363d",
    "axes.titlecolor": "#3fe0a8",
})
VERDE, VERMELHO, CINZA = "#3fe0a8", "#ff5c5c", "#7d8590"
# DRIVE não é redefinido aqui de propósito: quem define é a célula de
# setup, e uma variável de caminho com duas origens é como se perde a
# noção de onde os arquivos foram parar.
print("palco configurado")

In [ ]:
pose = YOLO(f"{DRIVE}/00-pesos/yolo11n-pose.pt")
print("modelo de pose pronto")

In [ ]:
# ── a regra: o que e "braco levantado" ──
#
# O modelo devolve 17 pontos por pessoa (padrao COCO):
#   5 ombro esq   6 ombro dir   7 cotovelo esq  8 cotovelo dir
#   9 pulso esq  10 pulso dir
#
# Em imagem o eixo Y cresce PARA BAIXO: pulso acima do ombro = y MENOR.
#
# ARMADILHA REAL (custou uma demo quebrada em teste): ponto que o modelo nao
# enxerga volta como (0, 0) com confianca baixa — e (0,0) fica no TOPO da
# imagem, ou seja, um pulso escondido seria lido como "braco levantado".
# Por isso todo ponto passa por um piso de confianca antes de valer.

OMBRO_E, OMBRO_D, PULSO_E, PULSO_D = 5, 6, 9, 10
CONF_MIN = 0.5          # abaixo disso o ponto nao existe para nos

def braco_levantado(pontos, confs):
    """True se QUALQUER pulso visivel estiver acima do ombro do mesmo lado."""
    for ombro, pulso in ((OMBRO_E, PULSO_E), (OMBRO_D, PULSO_D)):
        if confs[ombro] < CONF_MIN or confs[pulso] < CONF_MIN:
            continue                      # ponto nao confiavel: ignora o lado
        if pontos[pulso][1] < pontos[ombro][1]:
            return True
    return False

### Numa imagem parada (o aquecimento)

In [ ]:
import glob, cv2, matplotlib.pyplot as plt
entradas = [e for e in sorted(glob.glob(f"{DRIVE}/03-pose/entrada/*"))
            if e.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))]
IMG = entradas[0] if entradas else "https://ultralytics.com/images/bus.jpg"

r = pose.predict(IMG, conf=.35, verbose=False)[0]
im = r.plot(line_width=4, kpt_radius=8)

pessoas = len(r.boxes)
cima = 0
if r.keypoints is not None and pessoas:
    xy = r.keypoints.xy.cpu().numpy()
    cf = r.keypoints.conf.cpu().numpy() if r.keypoints.conf is not None else None
    if cf is not None:
        cima = sum(braco_levantado(p, c) for p, c in zip(xy, cf))

print(f"pessoas: {pessoas}   braco para cima: {cima}   para baixo: {pessoas - cima}")
plt.figure(); plt.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); plt.axis("off")
plt.title(f"{pessoas} pessoas · {cima} com o braço para cima"); plt.tight_layout(); plt.show()

### O painel ao vivo

Roda sobre um vídeo da pasta `03-pose/entrada/`. O `track` mantém **o mesmo
número** em cima de cada pessoa entre os quadros — é isso que permite contar
*movimentos* em vez de só contar braços.

In [ ]:
# ── processa o video, conta movimentos e grava o resultado anotado ──
import cv2, glob, numpy as np
from collections import defaultdict

videos = [v for v in sorted(glob.glob(f"{DRIVE}/03-pose/entrada/*"))
          if v.lower().endswith((".mp4", ".mov", ".avi", ".mkv"))]
if not videos:
    print("Nenhum video em 03-pose/entrada/ — pule para a celula da webcam.")
else:
    VIDEO = videos[0]
    print("processando:", VIDEO)

    estado = {}                    # id -> braco estava levantado?
    movimentos = defaultdict(int)    # id -> ciclos completos
    serie = []                       # (frame, pessoas, cima) para o grafico

    cap = cv2.VideoCapture(VIDEO)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    saida = f"{DRIVE}/03-pose/saida/pose_anotado.mp4"
    vw = cv2.VideoWriter(saida, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

    n = 0
    for res in pose.track(VIDEO, stream=True, persist=True, conf=.35, verbose=False):
        n += 1
        frame = res.plot(line_width=3, kpt_radius=6)
        pessoas = len(res.boxes) if res.boxes is not None else 0
        cima = 0

        if res.keypoints is not None and pessoas and res.boxes.id is not None:
            ids = res.boxes.id.cpu().numpy().astype(int)
            xy = res.keypoints.xy.cpu().numpy()
            cf = res.keypoints.conf.cpu().numpy() if res.keypoints.conf is not None else None
            if cf is not None:
                for pid, p, c in zip(ids, xy, cf):
                    lev = braco_levantado(p, c)
                    cima += lev
                    # ciclo completo = subiu e depois desceu
                    if estado.get(pid) and not lev:
                        movimentos[pid] += 1
                    estado[pid] = lev

        serie.append((n, pessoas, cima))

        # painel desenhado no proprio quadro, em tamanho de palco
        total_mov = sum(movimentos.values())
        cv2.rectangle(frame, (0, 0), (w, 92), (13, 17, 23), -1)
        cv2.putText(frame, f"PESSOAS {pessoas}", (24, 62), cv2.FONT_HERSHEY_SIMPLEX, 1.9, (230, 237, 243), 4)
        cv2.putText(frame, f"BRACO CIMA {cima}", (int(w*.34), 62), cv2.FONT_HERSHEY_SIMPLEX, 1.9, (168, 224, 63), 4)
        cv2.putText(frame, f"MOVIMENTOS {total_mov}", (int(w*.68), 62), cv2.FONT_HERSHEY_SIMPLEX, 1.9, (92, 92, 255), 4)
        vw.write(frame)

    cap.release(); vw.release()
    print(f"{n} quadros · movimentos contados: {sum(movimentos.values())}")
    print("salvo em", saida)

In [ ]:
# ── a serie temporal: como a sala reagiu, quadro a quadro ──
import matplotlib.pyplot as plt
if "serie" in dir() and serie:
    f = [s[0] for s in serie]; p = [s[1] for s in serie]; c = [s[2] for s in serie]
    fig, ax = plt.subplots()
    ax.plot(f, p, lw=4, color=CINZA, label="pessoas na cena")
    ax.plot(f, c, lw=5, color=VERDE, label="com o braço para cima")
    ax.fill_between(f, c, color=VERDE, alpha=.18)
    ax.set_xlabel("quadro"); ax.set_ylabel("quantidade")
    ax.set_title("A sala, medida quadro a quadro")
    ax.legend(loc="upper left")
    plt.tight_layout(); plt.show()
else:
    print("rode a celula anterior com um video primeiro")

---

# 🔴 AO VIVO · o polichinelo

A partir daqui é webcam de verdade, em tempo real: esqueleto desenhado sobre
cada pessoa, contagem de quem está na cena, e **um contador de movimentos**.

**A regra do movimento:** braços sobem → braços descem → **conta 1**. É o ciclo
do polichinelo. Cada pessoa tem o próprio contador, porque o modelo mantém um
**ID** em cima de cada uma entre os quadros.

> Fala de palco: *"levantem e abaixem os braços. Reparem que ele não conta
> braço levantado — ele conta o movimento COMPLETO, e conta separado por
> pessoa. Isso não veio pronto: o modelo me dá pontos, e a regra do que conta
> como uma repetição fui eu que escrevi. É a mesma divisão de trabalho de
> qualquer projeto sério de IA."*

In [ ]:
# ── motor de webcam ao vivo (leia o comentário: é o truque da demo) ──
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import cv2, numpy as np, PIL.Image, io, time

def iniciar_webcam(largura=640, altura=480):
    # cria o video no navegador + a camada de overlay por cima dele
    display(Javascript('''
      var video, div = null, stream, imgElement, labelElement, captureCanvas;
      var pendingResolve = null, shutdown = false;
      var LARG = %d, ALT = %d;

      function removeDom() {
        if (stream) stream.getVideoTracks()[0].stop();
        if (video) video.remove();
        if (div) div.remove();
        video = null; div = null; stream = null;
        imgElement = null; captureCanvas = null; labelElement = null;
      }

      function onAnimationFrame() {
        if (!shutdown) window.requestAnimationFrame(onAnimationFrame);
        if (pendingResolve) {
          var result = "";
          if (!shutdown) {
            captureCanvas.getContext('2d').drawImage(video, 0, 0, LARG, ALT);
            result = captureCanvas.toDataURL('image/jpeg', 0.75);
          }
          var lp = pendingResolve;
          pendingResolve = null;
          lp(result);
        }
      }

      async function criarDom() {
        if (div !== null) return stream;

        div = document.createElement('div');
        div.style.border = '2px solid #3fe0a8';
        div.style.padding = '3px';
        div.style.width = '100%%';
        div.style.maxWidth = '900px';
        div.style.borderRadius = '10px';
        document.body.appendChild(div);

        var parar = document.createElement('div');
        parar.innerHTML = '&#9632; clique aqui para encerrar';
        parar.style.cssText = 'cursor:pointer;background:#3fe0a8;color:#06231a;' +
          'font-weight:700;padding:10px 16px;border-radius:8px;text-align:center;' +
          'font-family:system-ui,sans-serif;font-size:18px';
        div.appendChild(parar);
        parar.onclick = function() { shutdown = true; };

        video = document.createElement('video');
        video.style.display = 'block';
        video.style.width = '100%%';
        video.setAttribute('playsinline', '');
        video.onclick = function() { shutdown = true; };

        stream = await navigator.mediaDevices.getUserMedia(
          {video: {width: LARG, height: ALT}});
        div.appendChild(video);
        video.srcObject = stream;
        await video.play();

        // a camada que recebe o resultado do modelo, por cima do vídeo
        imgElement = document.createElement('img');
        imgElement.style.position = 'absolute';
        imgElement.style.zIndex = 1;
        imgElement.style.pointerEvents = 'none';
        imgElement.onclick = function() { shutdown = true; };
        div.appendChild(imgElement);

        labelElement = document.createElement('div');
        labelElement.style.cssText = 'font-family:system-ui,sans-serif;' +
          'font-size:20px;color:#e6edf3;padding:8px 4px';
        div.appendChild(labelElement);

        captureCanvas = document.createElement('canvas');
        captureCanvas.width = LARG;
        captureCanvas.height = ALT;
        window.requestAnimationFrame(onAnimationFrame);

        google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
        return stream;
      }

      async function quadro(rotulo, overlay) {
        if (shutdown) { removeDom(); shutdown = false; return ''; }
        stream = await criarDom();
        if (rotulo != "") labelElement.innerHTML = rotulo;
        if (overlay != "") {
          var r = video.getClientRects()[0];
          imgElement.style.top = r.top + "px";
          imgElement.style.left = r.left + "px";
          imgElement.style.width = r.width + "px";
          imgElement.style.height = r.height + "px";
          imgElement.src = overlay;
        }
        var result = await new Promise(function(resolve) { pendingResolve = resolve; });
        shutdown = false;
        return {'img': result};
      }
    ''' % (largura, altura)))


def _para_imagem(resposta):
    # base64 do navegador -> imagem BGR do OpenCV
    if not resposta:
        return None
    dados = b64decode(resposta.split(',')[1])
    arr = np.frombuffer(dados, dtype=np.uint8)
    return cv2.imdecode(arr, flags=1)


def _para_overlay(rgba):
    # array RGBA -> data URI PNG, para o navegador sobrepor ao video
    img = PIL.Image.fromarray(rgba, 'RGBA')
    buf = io.BytesIO()
    img.save(buf, format='png')
    return 'data:image/png;base64,' + b64encode(buf.getvalue()).decode('utf-8')


def rodar_ao_vivo(processa, largura=640, altura=480, rotulo_inicial='iniciando…'):
    # Laco principal.
    #
    # `processa(frame_bgr, overlay_rgba)` recebe o quadro e uma tela RGBA
    # transparente do mesmo tamanho, desenha nela, e devolve o texto do
    # painel. Encerre clicando no botão verde (ou no próprio vídeo).
    iniciar_webcam(largura, altura)
    overlay = np.zeros([altura, largura, 4], dtype=np.uint8)
    envio = ''
    rotulo = rotulo_inicial
    n = 0
    t0 = time.time()
    try:
        while True:
            resposta = eval_js('quadro("{}", "{}")'.format(rotulo, envio))
            if not resposta:
                break
            frame = _para_imagem(resposta['img'])
            if frame is None:
                break

            overlay[:] = 0
            texto = processa(frame, overlay)

            n += 1
            fps = n / max(1e-6, time.time() - t0)
            rotulo = '{} &nbsp;·&nbsp; {:.1f} quadros/s'.format(texto, fps)
            envio = _para_overlay(overlay)
    except Exception as e:
        print('encerrado:', type(e).__name__, e)
    print('fim · {} quadros processados'.format(n))

In [ ]:
# ── a máquina de estados de cada pessoa ──────────────────────────────
#
# Um contador ingênuo ("está com o braço para cima?") dispara dezenas de
# vezes enquanto o braço fica erguido. O que interessa é a TRANSIÇÃO:
# só conta quando o braço sobe e DEPOIS desce — um ciclo fechado.
#
# A histerese (dois limiares em vez de um) existe porque o pulso treme em
# volta da linha do ombro: com um limiar só, um tremor de 2 px contaria
# cinco repetições que ninguém fez.

from collections import defaultdict

ALTURA_SOBE  = 0.04   # o pulso precisa passar BEM acima do ombro para "subir"
ALTURA_DESCE = 0.02   # e voltar BEM abaixo para "descer" — evita tremor

estado_pessoa = {}                     # id -> 'cima' | 'baixo'
repeticoes    = defaultdict(int)       # id -> ciclos completos

def altura_bracos(pontos, confs, altura_img):
    """Quanto o pulso está acima do ombro, em fração da altura da imagem.
    Positivo = acima. Devolve None quando não dá para confiar nos pontos."""
    melhor = None
    for ombro, pulso in ((OMBRO_E, PULSO_E), (OMBRO_D, PULSO_D)):
        if confs[ombro] < CONF_MIN or confs[pulso] < CONF_MIN:
            continue
        d = (pontos[ombro][1] - pontos[pulso][1]) / altura_img
        melhor = d if melhor is None else max(melhor, d)
    return melhor

def atualizar_repeticao(pid, d):
    """Devolve True quando um ciclo completo acabou de fechar."""
    if d is None:
        return False
    atual = estado_pessoa.get(pid, 'baixo')
    if atual == 'baixo' and d > ALTURA_SOBE:
        estado_pessoa[pid] = 'cima'
    elif atual == 'cima' and d < ALTURA_DESCE:
        estado_pessoa[pid] = 'baixo'
        repeticoes[pid] += 1          # fechou o ciclo: conta agora
        return True
    return False

print("regra carregada · sobe >", ALTURA_SOBE, "· desce <", ALTURA_DESCE)

### Antes do laço: o corpo em partes

O modelo devolve os mesmos **17 pontos** para todo mundo, numerados de 0 a 16, e
só isso. Que o ponto 5 ligado ao 7 ligado ao 9 seja **um braço** é uma tabela que
*nós* escrevemos — o modelo não sabe o que é braço.

A célula abaixo escreve essa tabela em quatro partes (**cabeça, braços, tronco,
pernas**), cada uma com a sua cor, e ainda usa os cinco pontos da cabeça — nariz,
olhos e orelhas — para dizer se a pessoa está **de frente, de perfil ou de
costas**.

> Fala de palco: *"a cor não é enfeite. Ela mostra que a máquina não vê 'uma
> pessoa': vê partes articuladas, e eu é que digo quais pontos formam cada parte.
> Repare que ele sabe se você está olhando para ele — só pelos olhos e orelhas."*

In [ ]:
# ── as partes do corpo, cada uma com a sua cor ───────────────────────
#
# Os 17 pontos do padrao COCO, agrupados por parte:
#   cabeca  0 nariz  1 olho esq  2 olho dir  3 orelha esq  4 orelha dir
#   bracos  7 cotovelo esq  8 cotovelo dir  9 pulso esq  10 pulso dir
#   tronco  5 ombro esq  6 ombro dir  11 quadril esq  12 quadril dir
#   pernas  13 joelho esq  14 joelho dir  15 tornozelo esq  16 tornozelo dir
#
# CUIDADO COM A ORDEM DAS CORES: o overlay que vai para o navegador e RGBA
# (vermelho, verde, azul, opacidade) — nao BGR como no OpenCV puro. Cor
# invertida aqui e o erro mais comum: o ciano sai laranja e ninguem entende.

CABECA             = [(0, 1), (0, 2), (1, 3), (2, 4)]
MEMBROS_SUPERIORES = [(5, 7), (7, 9), (6, 8), (8, 10)]
TRONCO             = [(5, 6), (5, 11), (6, 12), (11, 12)]
MEMBROS_INFERIORES = [(11, 13), (13, 15), (12, 14), (14, 16)]

COR_CABECA   = (232, 204, 110, 255)   # dourado
COR_SUPERIOR = ( 62, 207, 207, 255)   # ciano
COR_TRONCO   = (255, 170,  80, 255)   # laranja
COR_INFERIOR = (150, 150, 255, 255)   # azul
COR_REALCE   = ( 63, 224, 168, 255)   # verde-agua: braco no alto, repeticao fechou
CINZA        = (185, 195, 205, 255)
BRANCO       = (255, 255, 255, 255)

# (nome curto p/ legenda, conexoes, cor, pontos que pertencem a parte)
PARTES = [
    ("cabeca", CABECA,             COR_CABECA,   (0, 1, 2, 3, 4)),
    ("bracos", MEMBROS_SUPERIORES, COR_SUPERIOR, (7, 8, 9, 10)),
    ("tronco", TRONCO,             COR_TRONCO,   (5, 6, 11, 12)),
    ("pernas", MEMBROS_INFERIORES, COR_INFERIOR, (13, 14, 15, 16)),
]

# ponto -> cor, para o circulo sair na cor da propria parte
COR_DO_PONTO = {i: cor for _, _, cor, pontos in PARTES for i in pontos}

NARIZ, OLHO_E, OLHO_D, ORELHA_E, ORELHA_D = 0, 1, 2, 3, 4

def orientacao_cabeca(confs):
    """So com os pontos da cabeca ja da para saber para onde a pessoa olha.
    Dois olhos visiveis = de frente. Um so = perfil. Nenhum = de costas."""
    olhos = sum(confs[p] >= CONF_MIN for p in (OLHO_E, OLHO_D))
    if olhos == 2:
        return "de frente"
    if olhos == 1 or confs[NARIZ] >= CONF_MIN:
        return "de perfil"
    return "de costas"

def ancora_da_etiqueta(pontos, confs):
    """Onde pendurar o numero da pessoa.

    ARMADILHA REAL: ponto que o modelo nao viu volta como (0, 0) — que e o
    CANTO da tela. Usar pts[0] direto faz a etiqueta de quem esta de costas
    (sem nariz) ir parar no canto, empilhada com a de todo mundo. Por isso
    procuramos o primeiro ponto do alto que realmente foi visto."""
    for p in (NARIZ, OLHO_E, OLHO_D, ORELHA_E, ORELHA_D, OMBRO_E, OMBRO_D):
        if confs[p] >= CONF_MIN:
            return int(pontos[p][0]), int(pontos[p][1])
    return None

print("4 partes carregadas ·", " · ".join(nome for nome, _, _, _ in PARTES))

In [ ]:
# ── o laço ao vivo ───────────────────────────────────────────────────
import cv2, numpy as np

def desenha_esqueleto(overlay, pts, cs):
    """Desenha uma pessoa, cada parte na sua cor."""
    for _, conexoes, cor, _ in PARTES:
        for a, b in conexoes:
            if cs[a] < CONF_MIN or cs[b] < CONF_MIN:
                continue
            cv2.line(overlay, tuple(pts[a].astype(int)), tuple(pts[b].astype(int)), cor, 3)

    # o pescoco: sem ele a cabeca fica flutuando solta acima do corpo.
    # liga pela orelha quando da; de frente, sem orelha, liga o nariz ao
    # meio dos ombros.
    ligou = False
    for orelha, ombro in ((ORELHA_E, OMBRO_E), (ORELHA_D, OMBRO_D)):
        if cs[orelha] >= CONF_MIN and cs[ombro] >= CONF_MIN:
            cv2.line(overlay, tuple(pts[orelha].astype(int)),
                     tuple(pts[ombro].astype(int)), COR_CABECA, 2)
            ligou = True
    if not ligou and cs[NARIZ] >= CONF_MIN and cs[OMBRO_E] >= CONF_MIN and cs[OMBRO_D] >= CONF_MIN:
        meio = ((pts[OMBRO_E] + pts[OMBRO_D]) / 2).astype(int)
        cv2.line(overlay, tuple(pts[NARIZ].astype(int)), tuple(meio), COR_CABECA, 2)

    # os pontos, na cor da parte. Os 5 da cabeca sao miudos e ficam perto:
    # circulo menor, senao viram um borrao dourado so.
    for i, (x, y) in enumerate(pts):
        if cs[i] < CONF_MIN:
            continue
        raio = 3 if i in (OLHO_E, OLHO_D, ORELHA_E, ORELHA_D) else 5
        cv2.circle(overlay, (int(x), int(y)), raio, COR_DO_PONTO.get(i, BRANCO), -1)

PAINEL_H = 54          # altura da faixa preta do topo

def desenha_legenda(overlay, w, h):
    """Cor sem legenda nao comunica no telao.

    Vai no RODAPE de proposito: o topo ja tem o painel, e e justamente onde
    fica a CABECA de quem esta perto da camera — legenda ali tapa a pessoa.
    Embaixo quase sempre so tem chao."""
    topo = h - 30
    cv2.rectangle(overlay, (0, topo), (w, h), (13, 17, 23, 190), -1)
    x = 14
    for nome, _, cor, _ in PARTES:
        cv2.rectangle(overlay, (x, topo + 9), (x + 15, topo + 21), cor, -1)
        cv2.putText(overlay, nome.upper(), (x + 21, topo + 21),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, CINZA, 1)
        largura = cv2.getTextSize(nome.upper(), cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)[0][0]
        x += 21 + largura + 22

def processa_pose(frame, overlay):
    h, w = frame.shape[:2]
    r = pose.track(frame, persist=True, conf=.4, verbose=False, imgsz=480)[0]

    pessoas = 0
    no_alto = 0
    de_frente = 0

    if r.keypoints is not None and r.boxes is not None and len(r.boxes):
        pessoas = len(r.boxes)
        ids = (r.boxes.id.cpu().numpy().astype(int) if r.boxes.id is not None
               else np.arange(pessoas))
        xy = r.keypoints.xy.cpu().numpy()
        cf = (r.keypoints.conf.cpu().numpy() if r.keypoints.conf is not None
              else np.ones(xy.shape[:2]))

        for pid, pts, cs in zip(ids, xy, cf):
            desenha_esqueleto(overlay, pts, cs)

            d = altura_bracos(pts, cs, h)
            fechou = atualizar_repeticao(pid, d)
            esta_cima = estado_pessoa.get(pid) == 'cima'
            if esta_cima:
                no_alto += 1

            olhar = orientacao_cabeca(cs)
            if olhar == "de frente":
                de_frente += 1

            # etiqueta da pessoa, pendurada no ponto mais alto que o modelo viu
            ancora = ancora_da_etiqueta(pts, cs)
            if ancora is not None:
                cx, cy = ancora
                # nunca sobe atras do painel: quem esta perto da camera tem a
                # cabeca la em cima, e a etiqueta sumiria na faixa preta
                topo = max(PAINEL_H + 22, cy - 26)
                cor = COR_REALCE if esta_cima else CINZA
                cv2.putText(overlay, "#{}  {}x".format(pid, repeticoes[pid]),
                            (cx - 30, topo), cv2.FONT_HERSHEY_SIMPLEX, 0.8, cor, 2)
                # a leitura da cabeça, em letra miúda logo abaixo do número
                cv2.putText(overlay, olhar, (cx - 30, topo + 18),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.45, COR_CABECA, 1)
                if fechou:   # flash no quadro em que a repetição fecha
                    cv2.circle(overlay, (cx, max(20, topo - 34)), 16, COR_REALCE, -1)

    fizeram = sum(1 for v in repeticoes.values() if v > 0)
    total   = sum(repeticoes.values())

    # painel no topo do overlay, em tamanho de palco
    cv2.rectangle(overlay, (0, 0), (w, 54), (13, 17, 23, 210), -1)
    cv2.putText(overlay, "PESSOAS {}".format(pessoas), (14, 38),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (230, 237, 243, 255), 2)
    cv2.putText(overlay, "BRACOS CIMA {}".format(no_alto), (int(w*.30), 38),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, COR_REALCE, 2)
    cv2.putText(overlay, "MOVIMENTOS {}".format(total), (int(w*.66), 38),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (150, 150, 255, 255), 2)
    desenha_legenda(overlay, w, h)

    return ("<b>{}</b> pessoas &nbsp;·&nbsp; <b>{}</b> com o braço para cima "
            "&nbsp;·&nbsp; <b>{}</b> olhando para a câmera "
            "&nbsp;·&nbsp; <b>{}</b> movimentos de <b>{}</b> pessoas"
            .format(pessoas, no_alto, de_frente, total, fizeram))

# zera os contadores a cada nova rodada
estado_pessoa.clear(); repeticoes.clear()
rodar_ao_vivo(processa_pose, largura=640, altura=480,
              rotulo_inicial='autorize a câmera…')

### Se quiser recomeçar a contagem

Rode a célula abaixo e depois a do laço de novo — útil entre uma turma e outra,
ou quando a brincadeira acabar e você quiser um número limpo para a foto.

In [ ]:
estado_pessoa.clear()
repeticoes.clear()
print("contadores zerados")

### Uma foto só (plano B)

Se a câmera não abrir, ou se a conexão estiver ruim, esta célula tira **uma
foto** e analisa. Não é tão impressionante quanto o ao vivo, mas nunca falha —
e a frase de palco continua funcionando.

In [ ]:
# ── captura UM quadro da webcam e analisa ──
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import cv2, numpy as np, matplotlib.pyplot as plt

def foto(nome="webcam.jpg", qualidade=0.92):
    display(Javascript("""
      async function tirar(qualidade) {
        const div = document.createElement('div');
        const capturar = document.createElement('button');
        capturar.textContent = 'CLIQUE PARA CAPTURAR';
        capturar.style.cssText = 'font-size:22px;padding:14px 28px;margin:10px;cursor:pointer';
        div.appendChild(capturar);
        const video = document.createElement('video');
        video.style.display = 'block'; video.style.maxWidth = '100%';
        const stream = await navigator.mediaDevices.getUserMedia({video: true});
        document.body.appendChild(div); div.appendChild(video);
        video.srcObject = stream; await video.play();
        google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
        await new Promise((r) => capturar.onclick = r);
        const canvas = document.createElement('canvas');
        canvas.width = video.videoWidth; canvas.height = video.videoHeight;
        canvas.getContext('2d').drawImage(video, 0, 0);
        stream.getVideoTracks()[0].stop(); div.remove();
        return canvas.toDataURL('image/jpeg', qualidade);
      }
    """))
    dados = eval_js(f"tirar({qualidade})")
    binario = b64decode(dados.split(',')[1])
    with open(nome, 'wb') as f:
        f.write(binario)
    return nome

arq = foto()
r = pose.predict(arq, conf=.35, verbose=False)[0]
pessoas = len(r.boxes); cima = 0
if r.keypoints is not None and pessoas and r.keypoints.conf is not None:
    xy = r.keypoints.xy.cpu().numpy(); cf = r.keypoints.conf.cpu().numpy()
    cima = sum(braco_levantado(p, c) for p, c in zip(xy, cf))

plt.figure(); plt.imshow(cv2.cvtColor(r.plot(line_width=4, kpt_radius=8), cv2.COLOR_BGR2RGB))
plt.axis("off"); plt.title(f"{pessoas} pessoas · {cima} com o braço levantado")
plt.tight_layout(); plt.show()